# ToN-IoT Network: thí nghiệm 1 triệu dòng

Chạy các ô theo thứ tự. Điền URL GitHub của **đúng repository**; notebook không chứa URL hoặc token. Mỗi lần chỉ chọn một mô hình. Ô huấn luyện mặc định **tắt**; bật `RUN_TRAIN=True` khi đã kiểm tra dữ liệu và tài nguyên.

In [ ]:
import csv
import importlib
import json
import os
import platform
import re
import shutil
import stat
import subprocess
import sys
import tempfile
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

print("Python:", sys.version.split()[0], "| Hệ điều hành:", platform.platform())
print("Dung lượng trống /content (GiB):", round(shutil.disk_usage("/content").free / 1024**3, 2))
try:
    import psutil
    print("CPU:", psutil.cpu_count(), "| RAM còn trống (GiB):", round(psutil.virtual_memory().available / 1024**3, 2))
except ImportError:
    print("Chưa có psutil; bước cài thư viện sẽ bổ sung.")
print("nvidia-smi:", shutil.which("nvidia-smi") or "không có")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/ton_iot_network")
PROJECT_DIR = Path("/content/ton_iot_network")
DRIVE_SAMPLE_DIR = DRIVE_ROOT / "network_1m"
DRIVE_SAMPLE_ZIP = DRIVE_ROOT / "network_1m.zip"
LOCAL_SAMPLE_DIR = PROJECT_DIR / "colab_data" / "network_1m"
DRIVE_RESULT_DIR = DRIVE_ROOT / "results" / "network_1m"

GITHUB_REPO_URL = ""  # Điền URL GitHub của chính repository này; giữ trống sẽ dừng.
GITHUB_BRANCH = "refactor/unified-src"
UPDATE_SOURCE = True  # Chỉ cập nhật nhanh tiến trên đúng nhánh khi mã nguồn sạch.
print("Nguồn dữ liệu:", DRIVE_SAMPLE_DIR, "hoặc", DRIVE_SAMPLE_ZIP)
print("Kết quả:", DRIVE_RESULT_DIR)

In [ ]:
def git_command(*args):
    result = subprocess.run(["git", *map(str, args)], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f"Git thất bại ({' '.join(map(str, args))}): {result.stderr.strip() or result.stdout.strip()}")
    return result.stdout.strip()


def repository_url(value):
    return value.strip().rstrip("/").removesuffix(".git")


def prepare_repository(project_dir, repo_url, branch, update=True):
    """Clone hoặc cập nhật nhanh tiến; không đụng tới colab_data hay sửa đổi chưa lưu."""
    project_dir = Path(project_dir).resolve()
    if not repo_url.strip():
        raise ValueError("Hãy điền GITHUB_REPO_URL để kiểm tra đúng repository.")
    if project_dir.exists():
        if not (project_dir / ".git").exists():
            raise RuntimeError(f"{project_dir} đã tồn tại nhưng không phải Git repository; giữ nguyên thư mục này.")
        root = Path(git_command("-C", project_dir, "rev-parse", "--show-toplevel")).resolve()
        if root != project_dir:
            raise RuntimeError(f"Đường dẫn là thư mục con của repository khác: {root}")
        origin = git_command("-C", project_dir, "remote", "get-url", "origin")
        if repository_url(origin) != repository_url(repo_url):
            raise RuntimeError(f"Repository không khớp: origin={origin}; URL đã nhập={repo_url}")
        current = git_command("-C", project_dir, "symbolic-ref", "--short", "HEAD")
        if current != branch:
            raise RuntimeError(f"Nhánh hiện tại là {current}; cần {branch}. Không tự chuyển nhánh.")
        status = git_command("-C", project_dir, "status", "--porcelain", "--untracked-files=normal")
        changes = [
            line for line in status.splitlines()
            if not (line.startswith("?? ") and line[3:].replace("\\", "/").startswith("colab_data/"))
        ]
        if changes:
            raise RuntimeError("Mã nguồn có thay đổi chưa lưu; giữ nguyên và dừng cập nhật:\n" + "\n".join(changes[:20]))
        if update:
            git_command("-C", project_dir, "fetch", "origin", branch)
            remote_commit = git_command("-C", project_dir, "rev-parse", "FETCH_HEAD")
            git_command("-C", project_dir, "merge", "--ff-only", "FETCH_HEAD")
            if git_command("-C", project_dir, "rev-parse", "HEAD") != remote_commit:
                raise RuntimeError("HEAD chưa trùng commit vừa tải; kiểm tra nhánh và lịch sử Git.")
    else:
        git_command("clone", "--branch", branch, "--single-branch", repo_url, project_dir)
    source = project_dir / "src" / "ton_iot" / "training" / "experiments.py"
    if not source.is_file():
        raise RuntimeError(f"Thiếu {source}. Mã nguồn trên nhánh {branch} chưa đúng phiên bản.")
    commit = git_command("-C", project_dir, "rev-parse", "HEAD")
    print("Đúng repository và nhánh:", branch, "| commit:", commit)
    return commit

In [ ]:
PROJECT_COMMIT = prepare_repository(PROJECT_DIR, GITHUB_REPO_URL, GITHUB_BRANCH, update=UPDATE_SOURCE)

In [ ]:
requirements = PROJECT_DIR / "requirements.txt"
if not requirements.is_file():
    raise FileNotFoundError(f"Thiếu {requirements}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)

source_dir = (PROJECT_DIR / "src").resolve()
env = os.environ.copy()
env["PYTHONPATH"] = str(source_dir) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))
try:
    module = importlib.import_module("ton_iot.training.experiments")
    importlib.import_module("ton_iot.training.train")
    if Path(module.__file__).resolve() != source_dir / "ton_iot" / "training" / "experiments.py":
        raise RuntimeError(f"Đã import nhầm mã nguồn: {module.__file__}")
    check = subprocess.run(
        [sys.executable, "-c", "import ton_iot.training.experiments, ton_iot.training.train"],
        cwd=PROJECT_DIR, env=env, capture_output=True, text=True,
    )
    if check.returncode:
        raise RuntimeError(check.stderr.strip())
except Exception as error:
    raise RuntimeError(f"Không import được ton_iot từ src; dừng trước khi kiểm tra dữ liệu: {error}") from error
print("Import ton_iot thành công từ:", module.__file__)

## Chuẩn bị và kiểm tra bộ mẫu

Ưu tiên thư mục `network_1m/` trên Drive; nếu không có, dùng `network_1m.zip`. Chỉ sao chép hoặc giải nén vào ổ tạm Colab. Nếu bản tạm đã có, dùng lại và kiểm tra; không ghi đè bản trên Drive.

In [ ]:
def check_sample_files(root):
    root = Path(root)
    for relative in ("sampling_report.json", "metadata/train_fitted_rules.json"):
        if not (root / relative).is_file():
            raise FileNotFoundError(f"Bộ mẫu thiếu {root / relative}")
    for split in ("train", "validation", "test"):
        x_root = root / split / "X"
        x_parts = sorted(x_root.rglob("*.parquet"))
        if not x_parts:
            raise FileNotFoundError(f"Không có X Parquet: {x_root}")
        for x_path in x_parts:
            relative = x_path.relative_to(x_root)
            for kind in ("y", "metadata"):
                paired = root / split / kind / relative
                if not paired.is_file():
                    raise FileNotFoundError(f"Thiếu tệp ghép cặp: {paired}")


def safe_zip_entries(archive):
    for entry in archive.infolist():
        name = entry.filename.replace("\\", "/")
        parts = PurePosixPath(name).parts
        if name.startswith("/") or not parts or ".." in parts or ":" in parts[0]:
            raise ValueError(f"Đường dẫn ZIP không an toàn: {entry.filename}")
        if stat.S_ISLNK(entry.external_attr >> 16):
            raise ValueError(f"ZIP chứa liên kết mềm: {entry.filename}")


def prepare_sample(drive_root, local_sample):
    drive_root, local_sample = Path(drive_root), Path(local_sample)
    if local_sample.exists():
        check_sample_files(local_sample)
        print("Dùng lại bộ mẫu tạm đã có:", local_sample)
        return local_sample
    folder, archive_path = drive_root / "network_1m", drive_root / "network_1m.zip"
    if not folder.is_dir() and not archive_path.is_file():
        raise FileNotFoundError(f"Thiếu {folder} và {archive_path}; không có dữ liệu thay thế.")
    local_sample.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="network_1m_stage_", dir=local_sample.parent) as temporary:
        stage = Path(temporary)
        if folder.is_dir():
            check_sample_files(folder)
            candidate = stage / "network_1m"
            shutil.copytree(folder, candidate)
            print("Đã sao chép bộ mẫu từ thư mục Drive.")
        else:
            with zipfile.ZipFile(archive_path) as archive:
                safe_zip_entries(archive)
                archive.extractall(stage)
            if (stage / "sampling_report.json").is_file():
                candidate = stage
            elif (stage / "network_1m" / "sampling_report.json").is_file():
                candidate = stage / "network_1m"
            else:
                raise FileNotFoundError("ZIP phải chứa bộ mẫu ở gốc hoặc trong một thư mục network_1m.")
            print("Đã giải nén ZIP vào ổ tạm Colab.")
        check_sample_files(candidate)
        candidate.rename(local_sample)
    return local_sample

In [ ]:
LOCAL_SAMPLE_DIR = prepare_sample(DRIVE_ROOT, LOCAL_SAMPLE_DIR)
check = subprocess.run(
    [sys.executable, "-m", "ton_iot.training.experiments", "validate", "--data-dir", str(LOCAL_SAMPLE_DIR)],
    cwd=PROJECT_DIR, env=env, capture_output=True, text=True,
)
if check.returncode:
    raise RuntimeError("Kiểm tra bộ mẫu thất bại; không huấn luyện.\n" + (check.stderr or check.stdout)[-3000:])
validation_report = json.loads(check.stdout)
if validation_report.get("status") != "PASS":
    raise RuntimeError(f"Bộ mẫu chưa đạt: {validation_report.get('status')}")
sampling_report = json.loads((LOCAL_SAMPLE_DIR / "sampling_report.json").read_text(encoding="utf-8"))
print("KIỂM TRA BỘ MẪU:", validation_report["status"])
print("Số đặc trưng:", validation_report["features"])
for split in ("train", "validation", "test"):
    labels = validation_report["splits"][split]["labels"]
    total = validation_report["rows"][split]
    print(f"{split}: {total:,} dòng | bình thường {int(labels['0']):,} ({int(labels['0']) / total:.2%}) | tấn công {int(labels['1']):,} ({int(labels['1']) / total:.2%})")
print("X/y/metadata, nhãn, giá trị hữu hạn và thứ tự thời gian: PASS")

In [ ]:
RUN_GPU_CHECK = False  # Bật True trên runtime T4 trước khi chạy XGBoost GPU.
GPU_CHECK_OK = False
if RUN_GPU_CHECK:
    if not shutil.which("nvidia-smi"):
        raise RuntimeError("Không có nvidia-smi; chưa thể kiểm tra CUDA.")
    command = [
        sys.executable, "-m", "ton_iot.training.train", "--experiment", "network_1m",
        "--model", "xgboost", "--gpu-check", "--device", "gpu",
        "--data-dir", str(LOCAL_SAMPLE_DIR),
        "--output-dir", str(LOCAL_SAMPLE_DIR.parent / "gpu_check"),
        "--cpu-threads", "2",
    ]
    check = subprocess.run(command, cwd=PROJECT_DIR, env=env, capture_output=True, text=True)
    if check.returncode or "GPU CHECK PASS" not in check.stdout:
        raise RuntimeError("CUDA chưa đạt bài kiểm tra thực tế; không chạy XGBoost GPU.\n" + (check.stderr or check.stdout)[-3000:])
    GPU_CHECK_OK = True
    print("CUDA đã qua bài kiểm tra XGBoost nhỏ trên runtime hiện tại.")
else:
    print("Chưa chạy kiểm tra CUDA. XGBoost GPU sẽ bị chặn cho tới khi RUN_GPU_CHECK=True và kiểm tra đạt.")

## Huấn luyện một mô hình

Chọn `lightgbm/cpu`, `xgboost/gpu` hoặc `rf/cpu`. Giữ `RUN_TRAIN=False` khi chỉ muốn đọc kết quả đã có. Bài kiểm tra GPU ở trên là bắt buộc cho XGBoost GPU trong phiên làm việc hiện tại.

In [ ]:
MODELS = ("lightgbm", "xgboost", "rf")


def result_paths(result_root, model, run_id):
    if model not in MODELS or not re.fullmatch(r"\d{8}T\d{6}Z_[0-9a-f]{8}", run_id):
        raise ValueError("Tên mô hình hoặc mã lần chạy không hợp lệ.")
    root = Path(result_root) / model
    return {
        "manifest": root / "models" / model / run_id / "run_manifest.json",
        "metrics": root / "metrics" / run_id / model / "metrics.csv",
        "types": root / "metrics" / run_id / model / "attack_type_results.csv",
        "log": root / "logs" / f"{run_id}.log",
    }


def available_runs(result_root, model):
    root = Path(result_root) / model / "models" / model
    return sorted(path.parent.name for path in root.glob("*/run_manifest.json"))


def load_result(result_root, model, run_id):
    paths = result_paths(result_root, model, run_id)
    for key in ("manifest", "metrics", "types"):
        if not paths[key].is_file():
            raise FileNotFoundError(f"Thiếu kết quả {key}: {paths[key]}")
    manifest = json.loads(paths["manifest"].read_text(encoding="utf-8"))
    if (manifest.get("run_id"), manifest.get("model"), manifest.get("experiment")) != (run_id, model, "network_1m"):
        raise ValueError("Manifest không thuộc mô hình/lần chạy network_1m đã chọn.")
    with paths["metrics"].open(encoding="utf-8-sig", newline="") as handle:
        rows = list(csv.DictReader(handle))
    metrics = {row["split"]: row for row in rows if row["run_id"] == run_id and row["model"] == model}
    if set(metrics) != {"validation", "test"}:
        raise ValueError(f"Metrics thiếu validation/test cho {model}, run {run_id}.")
    with paths["types"].open(encoding="utf-8-sig", newline="") as handle:
        types = [row for row in csv.DictReader(handle) if row["run_id"] == run_id and row["model"] == model]
    if not types:
        raise ValueError(f"Thiếu kết quả theo loại tấn công cho {model}, run {run_id}.")
    return {"manifest": manifest, "metrics": metrics, "types": types, "paths": paths}


def show_result(result):
    manifest = result["manifest"]
    config = manifest.get("config", {})
    requested_gpu = config.get("device") == "cuda" or config.get("device_type") == "gpu"
    gpu_memory = manifest.get("peak_gpu_used_gib")
    gpu_note = ("không (CPU)" if not requested_gpu else
                f"có cấu hình GPU; ghi nhận {gpu_memory} GiB GPU" if gpu_memory else
                "có cấu hình GPU, mức sử dụng GPU chưa xác minh")
    print(f"Mô hình: {manifest['model']} | run: {manifest['run_id']}")
    print(f"Train thực tế: {manifest['train_rows_used']:,} dòng | đặc trưng: {manifest['feature_count']}")
    print(f"Thời gian huấn luyện: {manifest['fit_seconds']} giây | RAM đỉnh tiến trình: {manifest.get('peak_rss_gib')} GiB | GPU: {gpu_note}")
    print("Đường dẫn kết quả:", result["paths"]["manifest"].parents[3])
    import pandas as pd
    columns = [
        ("split", "Tập"), ("accuracy", "Độ chính xác"), ("f1_normal", "F1 bình thường"),
        ("f1_attack", "F1 tấn công"), ("tn", "Bình thường đúng"), ("fp", "Báo nhầm"),
        ("fn", "Tấn công bỏ sót"), ("tp", "Tấn công phát hiện"),
        ("false_positive_rate", "Tỷ lệ báo nhầm"),
    ]
    rows = []
    for split in ("validation", "test"):
        item = result["metrics"][split]
        rows.append({title: (round(float(item[key]) * 100, 3) if key in
                     {"accuracy", "f1_normal", "f1_attack", "false_positive_rate"} else
                     int(item[key]) if key in {"tn", "fp", "fn", "tp"} else split)
                     for key, title in columns})
    display(pd.DataFrame(rows))
    display(pd.DataFrame(result["types"])[["split", "type", "rows", "detected", "missed", "recall"]])

In [ ]:
MODEL = "lightgbm"  # "lightgbm", "xgboost" hoặc "rf"
DEVICE = "cpu"      # "gpu" chỉ với XGBoost và sau khi GPU check đạt
MAX_RAM_GIB = 10.0
RUN_TRAIN = False    # Chỉ đổi thành True khi muốn bắt đầu huấn luyện.
TRAINED_RUN_ID = None
TRAINED_MODEL = None

if RUN_TRAIN:
    if MODEL not in MODELS or (MODEL, DEVICE) not in {
        ("lightgbm", "cpu"), ("xgboost", "gpu"), ("rf", "cpu")
    }:
        raise ValueError("Chọn một trong lightgbm/cpu, xgboost/gpu, rf/cpu.")
    if validation_report.get("status") != "PASS":
        raise RuntimeError("Bộ mẫu chưa kiểm tra đạt; không huấn luyện.")
    if MODEL == "xgboost" and not GPU_CHECK_OK:
        raise RuntimeError("Cần chạy và đạt ô kiểm tra GPU trước khi huấn luyện XGBoost GPU.")
    result_dir = DRIVE_RESULT_DIR / MODEL
    command = [
        sys.executable, "-m", "ton_iot.training.train",
        "--experiment", "network_1m", "--model", MODEL, "--train-mode", "full",
        "--data-dir", str(LOCAL_SAMPLE_DIR), "--output-dir", str(result_dir),
        "--device", DEVICE, "--cpu-threads", "2", "--max-ram-gib", str(MAX_RAM_GIB),
    ]
    print("Bắt đầu huấn luyện:", MODEL, "|", DEVICE)
    process = subprocess.Popen(command, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    completed_run = None
    for line in process.stdout:
        print(line, end="")
        match = re.search(r"^RUN COMPLETE: (\d{8}T\d{6}Z_[0-9a-f]{8});", line)
        if match:
            completed_run = match.group(1)
    if process.wait() != 0 or completed_run is None:
        raise RuntimeError("Huấn luyện thất bại hoặc chưa hoàn thành; kiểm tra log trong thư mục kết quả.")
    trained_result = load_result(DRIVE_RESULT_DIR, MODEL, completed_run)
    TRAINED_RUN_ID, TRAINED_MODEL = completed_run, MODEL
    print("ĐÃ HUẤN LUYỆN VÀ LƯU ĐỦ KẾT QUẢ")
    show_result(trained_result)
else:
    print("Chưa huấn luyện. Để đọc lại kết quả cũ, chạy ô tiếp theo.")

## Đọc lại kết quả, không cần huấn luyện

Nếu một mô hình có nhiều lần chạy, điền chính xác `RESULT_RUN_ID`. Nếu vừa chạy xong mô hình đó, notebook ưu tiên mã lần chạy vừa hoàn thành.

In [ ]:
RESULT_MODEL = "lightgbm"
RESULT_RUN_ID = ""  # Ví dụ: "20261025T121329Z_a016b31f"; để trống khi chỉ có một run.
runs = available_runs(DRIVE_RESULT_DIR, RESULT_MODEL)
print("Các lần chạy có manifest:", runs or "chưa có")
chosen = RESULT_RUN_ID or (TRAINED_RUN_ID if TRAINED_MODEL == RESULT_MODEL else "")
if not chosen and len(runs) == 1:
    chosen = runs[0]
if not chosen:
    print("Chưa thể chọn kết quả. Nếu có nhiều lần chạy, điền RESULT_RUN_ID chính xác; không tự chọn file mới nhất.")
else:
    loaded_result = load_result(DRIVE_RESULT_DIR, RESULT_MODEL, chosen)
    show_result(loaded_result)

## Biểu đồ từ kết quả đã lưu

Ô này chỉ đọc các tệp thật. Nếu một mô hình chưa có kết quả hoặc có nhiều lần chạy mà chưa chọn mã, notebook thông báo và bỏ qua mô hình đó. Đặt `EXPORT_CHARTS=True` để lưu PNG vào `results/network_1m/charts/`; tên tệp mới có dấu thời gian, không ghi đè biểu đồ cũ.

In [ ]:
def plot_saved_results(sample_report, result_root, selected_runs=None, output_dir=None,
                       confusion_model="lightgbm"):
    import matplotlib.pyplot as plt
    import numpy as np

    selected_runs = selected_runs or {}
    results = {}
    for model in MODELS:
        runs = available_runs(result_root, model)
        chosen = selected_runs.get(model, "")
        if not chosen and len(runs) == 1:
            chosen = runs[0]
        if not chosen:
            print(f"{model}: chưa có kết quả hoặc có nhiều run; điền CHART_RUN_IDS để chọn. Có: {runs}")
            continue
        try:
            results[model] = load_result(result_root, model, chosen)
        except (FileNotFoundError, ValueError) as error:
            print(f"{model}: bỏ qua vì {error}")

    output_dir = Path(output_dir) if output_dir else None
    if output_dir:
        output_dir.mkdir(parents=True, exist_ok=True)
    tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
    saved = []

    def finish(fig, name):
        fig.patch.set_facecolor("white")
        fig.tight_layout()
        if output_dir:
            path = output_dir / f"{name}_{tag}.png"
            fig.savefig(path, dpi=180, facecolor="white")
            saved.append(path)
            print("Đã lưu:", path)
        plt.show()
        plt.close(fig)

    sample = sample_report.get("sample", {})
    if all(split in sample for split in ("train", "validation", "test")):
        splits = ("train", "validation", "test")
        normal = [int(sample[s]["labels"]["0"]) for s in splits]
        attack = [int(sample[s]["labels"]["1"]) for s in splits]
        totals = [a + b for a, b in zip(normal, attack)]
        x = np.arange(3)
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.bar(x, np.array(normal) / totals * 100, color="#444444", label="Bình thường")
        ax.bar(x, np.array(attack) / totals * 100, bottom=np.array(normal) / totals * 100,
               color="#bbbbbb", label="Tấn công")
        ax.set(xticks=x, xticklabels=["Train", "Validation", "Test"], ylabel="Tỷ lệ (%)",
               title="Tỷ lệ nhãn trong ba tập dữ liệu", ylim=(0, 100))
        ax.legend()
        finish(fig, "ty_le_nhan")
    else:
        print("Thiếu báo cáo lấy mẫu thật; bỏ qua biểu đồ tỷ lệ nhãn.")

    if not results:
        print("Chưa có kết quả huấn luyện hợp lệ; không tạo biểu đồ so sánh hoặc ma trận nhầm lẫn.")
        return saved

    names = list(results)
    x = np.arange(len(names))
    width = 0.34

    def paired_chart(field, title, ylabel, filename):
        fig, ax = plt.subplots(figsize=(max(6, len(names) * 2.2), 4))
        for index, split in enumerate(("validation", "test")):
            values = [100 * float(results[name]["metrics"][split][field]) for name in names]
            ax.bar(x + (index - 0.5) * width, values, width,
                   color=("#444444" if index == 0 else "#bbbbbb"), label=split.capitalize())
        ax.set(xticks=x, xticklabels=names, ylabel=ylabel, title=title, ylim=(0, 105))
        ax.legend()
        finish(fig, filename)

    paired_chart("accuracy", "Độ chính xác theo mô hình", "Độ chính xác (%)", "do_chinh_xac")
    fig, ax = plt.subplots(figsize=(max(6, len(names) * 2.2), 4))
    for index, (field, label) in enumerate((("f1_normal", "Bình thường"), ("f1_attack", "Tấn công"))):
        values = [100 * float(results[name]["metrics"]["test"][field]) for name in names]
        ax.bar(x + (index - 0.5) * width, values, width,
               color=("#444444" if index == 0 else "#bbbbbb"), label=label)
    ax.set(xticks=x, xticklabels=names, ylabel="F1 (%)", title="F1 từng lớp trên tập test", ylim=(0, 105))
    ax.legend()
    finish(fig, "f1_tung_lop")
    paired_chart("false_positive_rate", "Tỷ lệ cảnh báo sai", "Bình thường bị báo nhầm (%)", "canh_bao_sai")

    if confusion_model in results:
        row = results[confusion_model]["metrics"]["test"]
        matrix = np.array([[int(row["tn"]), int(row["fp"])], [int(row["fn"]), int(row["tp"])]])
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.imshow(matrix, cmap="Greys", vmin=0)
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{matrix[i, j]:,}", ha="center", va="center",
                        color="white" if matrix[i, j] > matrix.max() / 2 else "black")
        ax.set(xticks=[0, 1], xticklabels=["Bình thường", "Tấn công"],
               yticks=[0, 1], yticklabels=["Bình thường", "Tấn công"],
               xlabel="Dự đoán", ylabel="Nhãn thật",
               title=f"Ma trận nhầm lẫn trên test: {confusion_model}")
        finish(fig, f"ma_tran_{confusion_model}")
    else:
        print(f"Không có kết quả {confusion_model} để vẽ ma trận nhầm lẫn.")
    return saved

In [ ]:
EXPORT_CHARTS = False
CHART_RUN_IDS = {}  # Nếu có nhiều run, ví dụ {"lightgbm": "20261025T121329Z_a016b31f"}.
CONFUSION_MODEL = "lightgbm"
if TRAINED_RUN_ID and TRAINED_MODEL not in CHART_RUN_IDS:
    CHART_RUN_IDS[TRAINED_MODEL] = TRAINED_RUN_ID
saved_charts = plot_saved_results(
    sampling_report, DRIVE_RESULT_DIR, selected_runs=CHART_RUN_IDS,
    output_dir=DRIVE_RESULT_DIR / "charts" if EXPORT_CHARTS else None,
    confusion_model=CONFUSION_MODEL,
)